# Parkinson's keystroke EDA

This notebook explains why a dynamic feature such as MACD is reasonable. It imports the canonical code from `src/` and does not duplicate parsing or feature definitions.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[0]))
import numpy as np
import matplotlib.pyplot as plt
from src.data_loading import load_processed
from src.eda import subject_summary_rows


In [ ]:
sessions = load_processed()
subject_rows = subject_summary_rows(sessions)
print('sessions:', len(sessions))
print('subjects:', len(subject_rows))
print('cohorts:', sorted({s.cohort for s in sessions}))


## Scientific motivation

Parkinsonian motor impairment may affect the consistency of successive key presses. We therefore compare controls and PD subjects using ordinary hold-time summaries, the absolute change between consecutive hold times, and the corridor-restricted MACD feature. All comparisons are at the subject level so subjects with longer recordings do not dominate the plots.

In [ ]:
def boxplot(field, ylabel, title):
    values = [[r[field] for r in subject_rows if r['label_name']==label] for label in ['Control','PD']]
    plt.figure(figsize=(6,4)); plt.boxplot(values, tick_labels=['Control','PD'], showmeans=True)
    plt.ylabel(ylabel); plt.title(title); plt.show()

boxplot('mean_hold', 'Subject mean hold time (seconds)', 'Hold time by label')


In [ ]:
boxplot('mean_abs_consecutive_change', 'Mean absolute consecutive change (seconds)', 'Consecutive hold-time change by label')


## Hold time versus volatility

Each point is one subject. The x-axis is average hold time and the y-axis is average absolute change between consecutive hold times within sessions. Color shows control versus PD; marker shape shows cohort.

In [ ]:
colors = {'Control':'#2c7fb8', 'PD':'#d95f0e'}
markers = {'MIT-CS1PD':'o', 'MIT-CS2PD':'s'}
plt.figure(figsize=(7,5))
for row in subject_rows:
    plt.scatter(row['mean_hold'], row['mean_abs_consecutive_change'], color=colors[row['label_name']], marker=markers[row['cohort']], alpha=0.8, edgecolor='white', linewidth=0.5)
for label, color in colors.items(): plt.scatter([], [], color=color, marker='o', label=label)
for cohort, marker in markers.items(): plt.scatter([], [], color='gray', marker=marker, label=cohort)
plt.xlabel('Subject mean hold time (seconds)')
plt.ylabel('Mean absolute consecutive change (seconds)')
plt.title('Hold time versus timing volatility')
plt.legend(title='Color = label; marker = cohort')
plt.show()


## MACD feature comparison

MACD restricts consecutive changes to the upper hold-time corridor, defined per session by the 60th percentile through the maximum. This is the feature used in the replication model.

In [ ]:
boxplot('macd', 'Subject MACD (seconds)', 'MACD by label')

for cohort in sorted({r['cohort'] for r in subject_rows}):
    for label, name in [(0,'Control'),(1,'PD')]:
        vals=[r['macd'] for r in subject_rows if r['cohort']==cohort and r['label']==label]
        print(cohort, name, 'n=',len(vals), 'mean MACD=',round(float(np.mean(vals)),6), 'median=',round(float(np.median(vals)),6))


## Interpretation and caution

If PD subjects show larger or more variable consecutive hold-time changes and the corridor-restricted MACD separates the labels, that supports using MACD as an interpretable candidate feature. These are descriptive associations, not proof of causality or clinical validity. The final AUC is computed by the reproducible evaluation script with subject-safe cross-cohort testing.